## 데이터 불러오기

In [ ]:
# 라이브러리
import pandas as pd
import numpy as np
import re

In [3]:
# 데이터 로드
df = pd.read_csv("musinsastandard_reviews_raw.csv")
df_products = pd.read_csv("musinsastandard_woman_top50.csv")


print(f"리뷰 수: {len(df)}건")
print(f"컬럼: {df.columns.tolist()}")
print(df.head(3))

리뷰 수: 28216건
컬럼: ['goods_no', 'review_no', 'content', 'grade', 'option_size', 'size_feedback', 'create_date', 'review_sex', 'height', 'weight']
   goods_no  review_no                                            content  \
0   6092190   83820239  아이가 입울거라서 사이즈 때문에 매장에서 입어보고 샀어요! \n이 계절에 딱 필요한...   
1   6092190   83817463                  색상이 너무 고급스럽고 예뻐요 맘에 들어서 좋아요 잘입을께요   
2   6092190   83817336  착용감이 좋습니다.\n재질이 부드럽고 주머니지퍼 모자 허리에 스트링이 있어\n편리합니다.   

   grade     option_size size_feedback                    create_date  \
0      5      04.화이트 · S          정사이즈  2026-04-20T10:14:21.000+09:00   
1      5  02.클라우디 블루 · M          조금 큼  2026-04-20T05:44:20.000+09:00   
2      5  02.클라우디 블루 · L          조금 큼  2026-04-20T04:48:00.000+09:00   

  review_sex  height  weight  
0         여성     152      45  
1         여성     162      65  
2         여성     165      62  


# 리뷰 데이터 전처리

## 중복 리뷰

In [ ]:
# 중복 확인
print(f"중복 전: {len(df)}건")
print(f"중복 review_no 수: {df['review_no'].duplicated().sum()}건")

중복 전: 28216건
중복 review_no 수: 0건


## 옵션 사이즈 → 컬러/사이즈로 분리

In [5]:
# "04.화이트 · S" → color: "화이트", size: "S"
def parse_option(option):
    if pd.isna(option):
        return None, None
    # · 기준으로 분리
    parts = str(option).split('·')
    if len(parts) == 2:
        # 앞부분: 숫자. 제거 후 색상 추출
        color = re.sub(r'^\d+\.', '', parts[0]).strip()
        size = parts[1].strip()
        return color, size
    return None, None

df[['color', 'size']] = df['option_size'].apply(
    lambda x: pd.Series(parse_option(x))
)

print(df[['option_size', 'color', 'size']].head(10))
print(f"\ncolor 결측: {df['color'].isna().sum()}건")
print(f"size 결측: {df['size'].isna().sum()}건")
print(f"\n사이즈 종류: {sorted(df['size'].dropna().unique())}")
print(f"색상 종류: {df['color'].dropna().unique()}")

      option_size    color size
0      04.화이트 · S      화이트    S
1  02.클라우디 블루 · M  클라우디 블루    M
2  02.클라우디 블루 · L  클라우디 블루    L
3  02.클라우디 블루 · L  클라우디 블루    L
4  03.라이트 그레이 · L  라이트 그레이    L
5  02.클라우디 블루 · L  클라우디 블루    L
6  02.클라우디 블루 · L  클라우디 블루    L
7  02.클라우디 블루 · S  클라우디 블루    S
8  03.라이트 그레이 · S  라이트 그레이    S
9  03.라이트 그레이 · S  라이트 그레이    S

color 결측: 23585건
size 결측: 23585건

사이즈 종류: ['22', '24', '25', '26', '27', '28', '29', '30', '32', '34', 'L', 'M', 'S', 'XL', 'XS']
색상 종류: ['화이트' '클라우디 블루' '라이트 그레이' '페일 핑크' '블랙' '베이지' '크림' '화이트/화이트' '화이트/블랙'
 '블랙/블랙' '가죽블랙/버클실버' '가죽블랙/버클골드' '가죽화이트/버클실버' '가죽내추럴/버클골드' '가죽카멜/버클실버'
 '가죽내추럴/버클실버' '다크 브라운/실버' '블랙/실버' '브라운/실버' '화이트/실버' '블랙/골드' '브라운']


In [6]:
# · 없는 케이스 확인
no_dot = df[df['color'].isna()]['option_size'].dropna().unique()
print("color 결측인 option_size 샘플:")
print(no_dot[:20])

color 결측인 option_size 샘플:
['26' '27' '29' '25' '30' '24' '28' 'XS' 'M' 'L' 'S' 'XL' 'FREE' '31' '34'
 '32' '22' '23']


- 사이즈만 있고 컬러 구분 없는 단일 컬러 상품이라 color가 결측으로 나옴

In [7]:
def parse_option(option):
    if pd.isna(option):
        return None, None
    parts = str(option).split('·')
    if len(parts) == 2:
        # 컬러 + 사이즈
        color = re.sub(r'^\d+\.', '', parts[0]).strip()
        size = parts[1].strip()
        return color, size
    else:
        # 사이즈만 있는 경우 → color는 상품명에서 이미 알 수 있음
        size = str(option).strip()
        return None, size

df[['color', 'size']] = df['option_size'].apply(
    lambda x: pd.Series(parse_option(x))
)

print(f"color 결측: {df['color'].isna().sum()}건 (단일 컬러 상품)")
print(f"size 결측: {df['size'].isna().sum()}건")
print(f"\n사이즈 종류: {sorted(df['size'].dropna().unique())}")

color 결측: 23585건 (단일 컬러 상품)
size 결측: 16건

사이즈 종류: ['22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '34', 'FREE', 'L', 'M', 'S', 'XL', 'XS']


In [8]:
# size 결측 확인
print(df[df['size'].isna()][['goods_no', 'option_size', 'content']].head(16))

       goods_no option_size                                            content
8768    1243054         NaN  데일리로 쓰고았는 벨트입니다. 무난합니다. 매일 써서 해졌지만 이정도면 약하진 않은...
13246   2978126         NaN  편하게 막 입고 다니기 좋습니다. 길이가 조금 더 길면 좋을 것 같아요. 단점은 통...
13247   2978126         NaN  편하게 막 입고 다니기 좋습니다. 길이가 조금 더 길면 좋을 것 같아요. 단점은 통...
13248   2978126         NaN  오프라인으로 입어보고 사이즈 없어서 온라인으로 샀는데 너무너무 만족합니다~!!!색감...
13249   2978126         NaN                  비오는날 입으려고 구매 하였어용 \n아주 편안하고 만족합니당
13250   2978126         NaN  블랙 구매하고 싶었지만 매장도 온라인에도 제 사이즈는 품절이라 아쉽지만 건메탈 구입...
13251   2978126         NaN  무신사 강남 스토어에서 피팅해보고 구매했어요.\n검정품절이라 건메탈 구입했는데 색상...
13252   2978126         NaN                  얇고 가볍고 좋아요. 허리 스트링으로 조절 가능해서 편해요.
13253   2978126         NaN        최고예요 이뻐서 깔별로 사고싶어용\n얇고 시원해서 여름에 입을 긴바지로 좋아요
13254   2978126         NaN             편하고 이쁘고 걸을때 부스럭소리도 안나고 좋아용\n깔별로 쟁이고싶네요
13255   2978126         NaN        편하고 힙하고 걸을때 부스럭소리도 안나요!\n이뻐서 오래오래 잘 입을거같습니다
21981   3323598         NaN  흰색 셔츠가 급하게 필요해서 구매했어요\n

## grade 문자열 → 숫자열 변환

In [9]:
# grade 현재 타입 확인
print(f"grade 타입: {df['grade'].dtype}")
print(f"grade 고유값: {df['grade'].unique()}")

# 숫자로 변환
df['grade'] = pd.to_numeric(df['grade'], errors='coerce')

print(f"\n변환 후 grade 타입: {df['grade'].dtype}")
print(f"grade 분포:\n{df['grade'].value_counts().sort_index()}")

grade 타입: int64
grade 고유값: [5 4 3 1 2 0]

변환 후 grade 타입: int64
grade 분포:
grade
0      152
1       50
2       72
3      573
4     3028
5    24341
Name: count, dtype: int64


In [ ]:
# 0점 리뷰 확인
"""
0점은 리뷰 작성할 때 별점 자체를 매기지 않은 후기
"""
print(df[df['grade'] == 0][['goods_no', 'content', 'size_feedback']].head(5))

      goods_no                                            content  \
5037   3758435  일단 원단이 탄탄하고 길이감도 긴것이 너무 좃마요.\n뒷트임이있어 활동에도 불편함이...   
5042   3758435  적당한 두께로 속도 안비치고 길이도 길어서 예뻤어요\n허리에 비해 골반이 커서 S했...   
5044   3758435  이거 m 사이즈 샀다가 불편해서 거금 6천냥 들여서 사이즈 라지로 바꿨습니다. 쬐금...   
5049   3758435  이뻐요 기장도 너무맘에들고 여름에입기에는 쫌 더운감이 있었는데 지금 날씨에 입기 딱...   
5053   3758435  평소에 맥시 스커트 좋아해서 즐겨 입는데 디자인이 마음에 들어서 구매했어요! 깔끔한...   

     size_feedback  
5037           NaN  
5042           NaN  
5044           NaN  
5049           NaN  
5053           NaN  


In [ ]:
# 별점 없는 리뷰 표시
df['has_grade'] = df['grade'] != 0
print(f"별점 있는 리뷰: {df['has_grade'].sum()}건")
print(f"별점 없는 리뷰: {(~df['has_grade']).sum()}건")

별점 있는 리뷰: 28064건
별점 없는 리뷰: 152건


## 날짜 파싱(parsing)

In [12]:
# 날짜 변환
df['create_date'] = pd.to_datetime(df['create_date'])
df['year'] = df['create_date'].dt.year
df['month'] = df['create_date'].dt.month

print(f"날짜 범위: {df['create_date'].min()} ~ {df['create_date'].max()}")
print(f"\n월별 리뷰 분포:\n{df.groupby(['year','month'])['review_no'].count()}")

날짜 범위: 2023-03-14 19:55:41+09:00 ~ 2026-04-20 10:53:15+09:00

월별 리뷰 분포:
year  month
2023  3          20
      4          17
      5           9
      6          19
      7          11
      8          16
      9          20
      10         92
      11        138
      12         71
2024  1          47
      2          58
      3         194
      4         337
      5         479
      6         419
      7         468
      8         352
      9         520
      10        795
      11        975
      12        678
2025  1         342
      2         343
      3         586
      4        1119
      5        2041
      6        2721
      7        1873
      8        1327
      9        1468
      10       2253
      11       2210
      12       1025
2026  1         460
      2         993
      3        1936
      4        1784
Name: review_no, dtype: int64


## BMI 컬럼 추가

In [13]:
# BMI = 몸무게 / (키/100)²
df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)
df['bmi'] = df['bmi'].round(1)

print(f"BMI 결측: {df['bmi'].isna().sum()}건")
print(f"BMI 기술통계:\n{df['bmi'].describe()}")

BMI 결측: 2809건
BMI 기술통계:
count    25407.000000
mean        23.213390
std        376.335989
min          0.000000
25%         18.900000
50%         20.200000
75%         22.000000
max      60000.000000
Name: bmi, dtype: float64


In [14]:
# 비정상 BMI 확인
print("BMI 0인 케이스:")
print(df[df['bmi'] == 0][['height', 'weight', 'bmi']].head(5))

print("\nBMI 100 이상 케이스:")
print(df[df['bmi'] >= 100][['height', 'weight', 'bmi']].head(10))

BMI 0인 케이스:
       height  weight  bmi
5114      168       0  0.0
9826      164       0  0.0
9827      164       0  0.0
9895      174       0  0.0
13275     164       0  0.0

BMI 100 이상 케이스:
      height  weight    bmi
262      100     100  100.0
514      100     100  100.0
1978      60      50  138.9
2312     100     100  100.0
2829     100     100  100.0
2830     100     100  100.0
2878     100     100  100.0
3214      63      51  128.5
3567     100     100  100.0
4003      40      40  250.0


In [15]:
"""
weight == 0 → 몸무게를 0으로 입력한 케이스
height == 100, weight == 100 → 기본값으로 입력한 케이스 (실제 체형 아님)
height == 40~60 → 키를 잘못 입력한 케이스
"""

# 정상 범위 필터링
# 키: 140~200cm, 몸무게: 30~150kg
before = len(df)

df['bmi'] = df['bmi'].where(
    (df['height'] >= 140) & (df['height'] <= 200) &
    (df['weight'] >= 30) & (df['weight'] <= 150),
    other=np.nan
)

# bmi 재계산
df['bmi'] = df['weight'].where(
    (df['height'] >= 140) & (df['height'] <= 200) &
    (df['weight'] >= 30) & (df['weight'] <= 150)
) / ((df['height'].where(
    (df['height'] >= 140) & (df['height'] <= 200) &
    (df['weight'] >= 30) & (df['weight'] <= 150)
) / 100) ** 2)

df['bmi'] = df['bmi'].round(1)

print(f"BMI 정상값: {df['bmi'].notna().sum()}건")
print(f"BMI 이상치→NaN 처리: {before - df['bmi'].notna().sum()}건")
print(f"\nBMI 기술통계:\n{df['bmi'].describe()}")

BMI 정상값: 25171건
BMI 이상치→NaN 처리: 3045건

BMI 기술통계:
count    25171.000000
mean        20.692110
std          2.883482
min          9.300000
25%         18.900000
50%         20.200000
75%         22.000000
max         64.100000
Name: bmi, dtype: float64


In [22]:
# 현재 상태 확인
print(f"height 이상치 (140 미만 or 200 초과): {((df['height'] < 140) | (df['height'] > 200)).sum()}건")
print(f"weight 이상치 (30 미만 or 150 초과): {((df['weight'] < 30) | (df['weight'] > 150)).sum()}건")

# 이상치 → NaN 처리
df['height'] = df['height'].where(
    (df['height'] >= 140) & (df['height'] <= 200)
)
df['weight'] = df['weight'].where(
    (df['weight'] >= 30) & (df['weight'] <= 150)
)

print(f"\n처리 후 height 결측: {df['height'].isna().sum()}건")
print(f"처리 후 weight 결측: {df['weight'].isna().sum()}건")
print(f"\nheight 기술통계:\n{df['height'].describe()}")

height 이상치 (140 미만 or 200 초과): 2921건
weight 이상치 (30 미만 or 150 초과): 2946건

처리 후 height 결측: 2921건
처리 후 weight 결측: 2946건

height 기술통계:
count    25286.000000
mean       163.597801
std          5.851554
min        140.000000
25%        160.000000
50%        163.000000
75%        167.000000
max        200.000000
Name: height, dtype: float64


## 리뷰 성별 확인

In [23]:
print(df['review_sex'].value_counts(dropna=False))

review_sex
여성     23934
NaN     2893
남성      1380
Name: count, dtype: int64


- 확인해보니까 대부분 여자인데 남자로 선택하고 리뷰를 작성한 것으로 추정됨
- 유지하되, 신뢰도 낮은 데이터로 보는 것이 좋아보임.

## Size 컬럼 타입 확인

In [24]:
# 숫자 사이즈 vs 문자 사이즈 구분
def size_type(size):
    if pd.isna(size):
        return None
    if str(size).isdigit():
        return '숫자형'  # 하의 (22, 24, 26...)
    else:
        return '문자형'  # 의류 (XS, S, M, L, XL, FREE)

df['size_type'] = df['size'].apply(size_type)

print(df['size_type'].value_counts(dropna=False))
print(f"\n카테고리별 사이즈 타입:")
print(df.groupby(['category', 'size_type'])['review_no'].count())

size_type
문자형     17951
숫자형     10240
None       16
Name: count, dtype: int64

카테고리별 사이즈 타입:
category  size_type
가방        문자형           241
상의        문자형          8408
스커트       문자형           746
아우터       문자형          5160
액세서리      숫자형          1999
하의        문자형          3396
          숫자형          8241
Name: review_no, dtype: int64


## 텍스트 클리닝

In [16]:
def clean_text(text):
    if pd.isna(text):
        return None
    # 이모지 제거
    text = re.sub(r'[^\w\s가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9]', ' ', text)
    # 여러 공백 → 단일 공백
    text = re.sub(r'\s+', ' ', text)
    # 앞뒤 공백 제거
    text = text.strip()
    return text

df['content_clean'] = df['content'].apply(clean_text)

# 너무 짧은 리뷰 확인 (5자 이하)
short_reviews = df[df['content_clean'].str.len() <= 5]
print(f"5자 이하 리뷰: {len(short_reviews)}건")
print(short_reviews['content_clean'].values)

5자 이하 리뷰: 3건
['슬랙스코디' '좋아요' '좋아요']


In [17]:
print(f"제거 전: {len(df)}건")
df = df[df['content_clean'].str.len() > 5]
print(f"제거 후: {len(df)}건")

제거 전: 28216건
제거 후: 28207건


# top50과 조인

In [18]:
# top50에서 필요한 컬럼만 선택
df_products_slim = df_products[['goods_no', 'goods_name', 'sale_price', 'sale_rate', 'review_count', 'review_score']]

# 병합
df = df.merge(df_products_slim, on='goods_no', how='left')

print(f"병합 후 컬럼: {df.columns.tolist()}")
print(f"\n병합 확인:\n{df[['goods_no', 'goods_name', 'content_clean']].head(3)}")
print(f"\ngoods_name 결측: {df['goods_name'].isna().sum()}건")

병합 후 컬럼: ['goods_no', 'review_no', 'content', 'grade', 'option_size', 'size_feedback', 'create_date', 'review_sex', 'height', 'weight', 'color', 'size', 'has_grade', 'year', 'month', 'bmi', 'content_clean', 'goods_name', 'sale_price', 'sale_rate', 'review_count', 'review_score']

병합 확인:
   goods_no                                     goods_name  \
0   6092190  [한소희 PICK] [UV 프로텍션] 우먼즈 시어 윈드브레이커 (5 colors)   
1   6092190  [한소희 PICK] [UV 프로텍션] 우먼즈 시어 윈드브레이커 (5 colors)   
2   6092190  [한소희 PICK] [UV 프로텍션] 우먼즈 시어 윈드브레이커 (5 colors)   

                                       content_clean  
0  아이가 입울거라서 사이즈 때문에 매장에서 입어보고 샀어요 이 계절에 딱 필요한 필수...  
1                  색상이 너무 고급스럽고 예뻐요 맘에 들어서 좋아요 잘입을께요  
2      착용감이 좋습니다 재질이 부드럽고 주머니지퍼 모자 허리에 스트링이 있어 편리합니다  

goods_name 결측: 0건


## 카테고리 분류

In [19]:
# 상품명 목록 확인
print(df['goods_name'].unique())

['[한소희 PICK] [UV 프로텍션] 우먼즈 시어 윈드브레이커 (5 colors)'
 '[한소희 PICK] 우먼즈 코튼 커브드 팬츠 (3 colors)' '우먼즈 오버사이즈 블레이저 [블랙]'
 '[한소희 PICK] 우먼즈 슬림 크루 넥 티셔츠 [화이트]' '[한소희 PICK] 우먼즈 베이식 블레이저 [블랙]'
 '우먼즈 베이식 크루 넥 티셔츠 2팩' '우먼즈 CP 맥시 스커트 [화이트]' '우먼즈 나일론 슬라우치 숄더백 [블랙]'
 '우먼즈 더블 니트 투 턱 와이드 라운지 스웨트 팬츠 [블랙]' '[한소희 PICK] 우먼즈 돌먼 필드 재킷 [크림]'
 '우먼즈 폴카 도트 윈드브레이커 재킷 [블랙]' '우먼즈 와이드 카고 팬츠 [블랙]'
 '[한소희 PICK] 우먼즈 커브드 데님 팬츠 [미디엄 인디고]' '우먼즈 오버사이즈 하이넥 윈드브레이커 재킷 [블랙]'
 '우먼즈 베이식 이탈리안 레더 벨트_23mm' '우먼즈 오버사이즈 블레이저 [미디엄 그레이]'
 '우먼즈 베이식 크루 넥 티셔츠 [화이트]' '우먼즈 슬림 크루 넥 티셔츠 2팩' '우먼즈 와이드 히든 밴딩 슬랙스 [블랙]'
 '우먼즈 파라슈트 팬츠 [건메탈]' '우먼즈 파라슈트 팬츠 [블랙]' '우먼즈 돌먼 필드 재킷 [라이트 그레이]'
 '우먼즈 데님 맥시 스커트 [딥 인디고]' '우먼즈 커브드 치노 팬츠 [베이지]'
 '우먼즈 신세틱 스웨이드 하이넥 블루종 재킷 [라이트 브라운]' '[쿨탠다드] 우먼즈 베이식 크루 넥 티셔츠 2팩'
 '우먼즈 와이드 데님 팬츠 [크림]' '우먼즈 싱글 하프 트렌치 코트 [크림]' '우먼즈 신세틱 레더 슬림 벨트_20mm'
 '[세은 PICK] 우먼즈 시어 후디드 윈드브레이커 재킷 [민트 그레이]'
 '우먼즈 더블 니트 투 턱 와이드 라운지 스웨트 팬츠 [미디엄 그레이]' '우먼즈 나일론 루치드 롱 스커트 [애쉬 그레이]'
 '우먼즈 하이 웨이스트 와이드 히든 밴딩 슬랙스 [블랙]' '[한소희 PICK] 우먼즈 슬림 핏 셔츠 [화이트]'
 '우먼즈 코튼 블루종 재킷

In [20]:
def categorize(name):
    name = str(name)
    if any(k in name for k in ['티셔츠', '셔츠']):
        return '상의'
    elif any(k in name for k in ['팬츠', '슬랙스', '데님']):
        return '하의'
    elif any(k in name for k in ['스커트']):
        return '스커트'
    elif any(k in name for k in ['블레이저', '재킷', '윈드브레이커', '코트', '블루종']):
        return '아우터'
    elif any(k in name for k in ['벨트']):
        return '액세서리'
    elif any(k in name for k in ['백', '백팩', '숄더백']):
        return '가방'
    else:
        return '기타'

df['category'] = df['goods_name'].apply(categorize)

print(df['category'].value_counts())
print(f"\n기타 항목 확인:")
print(df[df['category'] == '기타']['goods_name'].unique())

category
하의      11647
상의       8413
아우터      5160
액세서리     2000
스커트       746
가방        241
Name: count, dtype: int64

기타 항목 확인:
[]


In [25]:
print(f"최종 데이터: {len(df)}건, {len(df.columns)}개 컬럼")
print(f"컬럼 목록: {df.columns.tolist()}")

최종 데이터: 28207건, 24개 컬럼
컬럼 목록: ['goods_no', 'review_no', 'content', 'grade', 'option_size', 'size_feedback', 'create_date', 'review_sex', 'height', 'weight', 'color', 'size', 'has_grade', 'year', 'month', 'bmi', 'content_clean', 'goods_name', 'sale_price', 'sale_rate', 'review_count', 'review_score', 'category', 'size_type']


In [26]:
# 확인
print(df[['content', 'content_clean']].head(3))

                                             content  \
0  아이가 입울거라서 사이즈 때문에 매장에서 입어보고 샀어요! \n이 계절에 딱 필요한...   
1                  색상이 너무 고급스럽고 예뻐요 맘에 들어서 좋아요 잘입을께요   
2  착용감이 좋습니다.\n재질이 부드럽고 주머니지퍼 모자 허리에 스트링이 있어\n편리합니다.   

                                       content_clean  
0  아이가 입울거라서 사이즈 때문에 매장에서 입어보고 샀어요 이 계절에 딱 필요한 필수...  
1                  색상이 너무 고급스럽고 예뻐요 맘에 들어서 좋아요 잘입을께요  
2      착용감이 좋습니다 재질이 부드럽고 주머니지퍼 모자 허리에 스트링이 있어 편리합니다  
